# Deploy Final ML Model - CatBoost Classifier

# Load Reqireed Libraries For Model Deployment

In [1]:
# =============================================================================
# DEPLOYMENT LIBRARIES
# =============================================================================

import pandas as pd
import numpy as np

# Load the trained model
import pickle

# CatBoost is required when loading/using a CatBoost model
from catboost import CatBoostClassifier

# Explainable AI
import shap

# Import joblib for saving and loading trained machine learning models.
# Joblib is particularly efficient for serialising Python objects such as
# trained Scikit-learn and machine learning models.
import joblib

import pickle  # Import pickle to save and load trained machine learning models

# Path: creates and manages file/folder paths
from pathlib import Path

# JSON: saves and reads feature configuration
import json

# shutil: copies the trained CatBoost model
import shutil

# FastAPI: creates the REST API
from fastapi import FastAPI, HTTPException

# Pydantic: validates API request data
from pydantic import BaseModel

# Typing: allows flexible customer input dictionaries
from typing import Dict, Any




# Load Saved CatBoost Churn Model for Deployment

In [2]:
# =============================================================================
# LOAD THE SAVED CATBOOST CHURN MODEL
# =============================================================================

# Load the previously saved CatBoost model from the .pkl file
tuned_catboost = joblib.load(
    "CatBoost_churn_model.pkl"
)

# Confirm that the model has been loaded successfully
print("=== CatBoost churn model loaded successfully ===")

=== CatBoost churn model loaded successfully ===


# Automated FastAPI Deployment Folder Setup

In [3]:
# =============================================================================
# PAYSTONE CUSTOMER CHURN PREDICTION
# AUTOMATED FASTAPI DEPLOYMENT FOLDER SETUP
# =============================================================================
#
# PURPOSE
# -------
# This script automatically prepares the folder structure required to deploy
# the trained PayStone CatBoost customer churn model as a FastAPI application.
#
# The script does NOT retrain the machine learning model.
#
# It uses the already-trained model:
#
#     CatBoost_churn_model.pkl
#
# and prepares the following deployment structure:
#
#     paystone-churn-api/
#     │
#     ├── app.py
#     ├── requirements.txt
#     ├── Dockerfile
#     ├── .gitignore
#     │
#     └── model/
#         └── CatBoost_churn_model.pkl
#
# This folder can subsequently be connected to GitHub and deployed to Render.
# =============================================================================

# =============================================================================
# STEP 1: DEFINE THE PAYSTONE WORKING DIRECTORY
# =============================================================================
#
# This is the existing folder containing your PayStone project and trained
# CatBoost model.
#
# IMPORTANT:
# The 'r' before the path creates a raw string. This prevents Windows
# backslashes (\) from being interpreted as special characters.
# =============================================================================

working_dir = Path(
    r"C:\Users\EUGENE\Desktop\RISK MODELLING ANALYTICS\CUSTOMER ANALYTICS\PayStone _Explainable AI Bank Customer Churn Prediction"
)


# =============================================================================
# STEP 2: VERIFY THAT THE WORKING DIRECTORY EXISTS
# =============================================================================

print("=" * 80)
print("PAYSTONE FASTAPI DEPLOYMENT SETUP")
print("=" * 80)

print("\nChecking working directory...")

if not working_dir.exists():

    # Stop the script if the specified directory cannot be found.
    raise FileNotFoundError(
        "\nWorking directory was not found.\n"
        f"Expected location:\n{working_dir}"
    )

else:

    print("✓ Working directory confirmed.")
    print(f"  {working_dir}")


# =============================================================================
# STEP 3: DEFINE THE FASTAPI DEPLOYMENT DIRECTORY
# =============================================================================
#
# The new deployment folder will be created INSIDE your existing PayStone
# working directory.
#
# Result:
#
# PayStone _Explainable AI Bank Customer Churn Prediction/
# │
# └── paystone-churn-api/
#
# =============================================================================

project_dir = (
    working_dir / "paystone-churn-api"
)


# =============================================================================
# STEP : CREATE THE MODEL DIRECTORY
# =============================================================================
#4
# The trained CatBoost model will be stored inside:
#
# paystone-churn-api/model/
#
# 'parents=True' allows Python to create the parent directories if required.
#
# 'exist_ok=True' prevents an error if the folder already exists.
# =============================================================================

model_dir = (
    project_dir / "model"
)

model_dir.mkdir(
    parents=True,
    exist_ok=True
)

print("\n✓ Model directory created/verified:")
print(f"  {model_dir}")


# =============================================================================
# STEP 5: DEFINE THE SOURCE CATBOOST MODEL
# =============================================================================
#
# The existing model is expected to be located directly inside the PayStone
# working directory.
#
# Expected:
#
# ...\PayStone _Explainable AI Bank Customer Churn Prediction\
#     CatBoost_churn_model.pkl
#
# =============================================================================

source_model = (
    working_dir /
    "CatBoost_churn_model.pkl"
)


# =============================================================================
# STEP 6: DEFINE THE MODEL DESTINATION
# =============================================================================
#
# The model will be COPIED rather than moved.
#
# Therefore, your original model remains safely in the original project
# directory.
#
# New location:
#
# paystone-churn-api/
#     model/
#         CatBoost_churn_model.pkl
#
# =============================================================================

destination_model = (
    model_dir /
    "CatBoost_churn_model.pkl"
)


# =============================================================================
# STEP 7: CHECK WHETHER THE SOURCE MODEL EXISTS
# =============================================================================

print("\nChecking saved CatBoost model...")

if not source_model.exists():

    # If the model does not exist, stop the setup because the API cannot
    # operate without the trained model.
    raise FileNotFoundError(
        "\nCatBoost model was not found.\n"
        f"Expected location:\n{source_model}\n\n"
        "Please confirm that 'CatBoost_churn_model.pkl' exists "
        "in your PayStone working directory."
    )


print("✓ CatBoost model found.")
print(f"  {source_model}")


# =============================================================================
# STEP 8: COPY THE TRAINED CATBOOST MODEL
# =============================================================================
#
# shutil.copy2() copies both:
#
# 1. The model file
# 2. Its metadata such as modification time
#
# The original model is NOT deleted.
# =============================================================================

print("\nCopying CatBoost model...")

# If the destination does not exist, copy the model.
if not destination_model.exists():

    shutil.copy2(
        source_model,
        destination_model
    )

    print("✓ CatBoost model copied successfully.")

# If the destination already exists, do not unnecessarily overwrite it.
else:

    print(
        "✓ CatBoost model already exists in the deployment folder."
    )


# =============================================================================
# STEP 9: VERIFY THAT THE COPIED MODEL CAN BE LOADED
# =============================================================================
#
# This is an important deployment check.
#
# A model may exist as a file but still fail to load because of:
#
# - File corruption
# - Incorrect path
# - Missing Python dependencies
# - Incompatible serialization environment
#
# We therefore attempt to load the copied model using joblib.
# =============================================================================

print("\nTesting copied CatBoost model...")

try:

    deployment_model = joblib.load(
        destination_model
    )

    print(
        "✓ CatBoost model loaded successfully."
    )

    print(
        f"  Model type: {type(deployment_model)}"
    )

except Exception as error:

    raise RuntimeError(
        "\nThe CatBoost model exists but could not be loaded.\n"
        f"Error: {error}"
    )


# =============================================================================
# STEP 10: CREATE DEPLOYMENT FILES
# =============================================================================
#
# These files are required for the FastAPI + Docker + Render deployment.
#
# app.py
# -------
# Contains the FastAPI application and prediction endpoint.
#
# requirements.txt
# -----------------
# Contains the Python packages required by the API.
#
# Dockerfile
# ----------
# Defines how the application is packaged and run on Render.
#
# .gitignore
# ----------
# Prevents unnecessary/local files from being uploaded to GitHub.
# =============================================================================

deployment_files = [

    "app.py",

    "requirements.txt",

    "Dockerfile",

    ".gitignore"

]


print("\nCreating deployment files...")


for file_name in deployment_files:

    # Construct the complete path.
    file_path = (
        project_dir / file_name
    )

    # Only create the file if it doesn't already exist.
    #
    # This protects existing code from accidental overwriting.
    if not file_path.exists():

        file_path.touch()

        print(
            f"✓ Created: {file_name}"
        )

    else:

        print(
            f"✓ Already exists: {file_name}"
        )


# =============================================================================
# STEP 11: DISPLAY THE FINAL DEPLOYMENT STRUCTURE
# =============================================================================
#
# This section recursively searches the deployment folder and displays every
# file that has been created.
# =============================================================================

print("\n")
print("=" * 80)
print("PAYSTONE FASTAPI DEPLOYMENT STRUCTURE")
print("=" * 80)


print("\nDeployment project:")
print(
    project_dir
)


print("\nFiles:")


for path in sorted(
    project_dir.rglob("*")
):

    # Only display files, not directories.
    if path.is_file():

        relative_path = (
            path.relative_to(
                project_dir
            )
        )

        print(
            f"   ✓ {relative_path}"
        )


# =============================================================================
# STEP 12: CHECK REQUIRED DEPLOYMENT COMPONENTS
# =============================================================================
#
# The API deployment requires these five components:
#
# 1. app.py
# 2. requirements.txt
# 3. Dockerfile
# 4. .gitignore
# 5. CatBoost model
#
# This section confirms that all required components exist.
# =============================================================================

required_files = [

    project_dir / "app.py",

    project_dir / "requirements.txt",

    project_dir / "Dockerfile",

    project_dir / ".gitignore",

    destination_model

]


print("\n")
print("=" * 80)
print("DEPLOYMENT COMPONENT CHECK")
print("=" * 80)


all_components_available = True


for file_path in required_files:

    if file_path.exists():

        print(
            f"✓ FOUND: {file_path.name}"
        )

    else:

        print(
            f"✗ MISSING: {file_path.name}"
        )

        all_components_available = False


# =============================================================================
# STEP 13: FINAL STATUS
# =============================================================================

print("\n")
print("=" * 80)


if all_components_available:

    print(
        "✓ PAYSTONE FASTAPI DEPLOYMENT FOLDER READY"
    )

    print("\nDeployment directory:")
    print(project_dir)

    print("\nNext steps:")
    print("1. Build the FastAPI application in app.py")
    print("2. Add dependencies to requirements.txt")
    print("3. Configure Dockerfile")
    print("4. Test the API locally")
    print("5. Create GitHub repository")
    print("6. Push deployment folder to GitHub")
    print("7. Connect GitHub repository to Render")
    print("8. Deploy the FastAPI service")


else:

    print(
        "✗ DEPLOYMENT SETUP INCOMPLETE"
    )

    print(
        "Please check the missing files above."
    )


print("=" * 80)

PAYSTONE FASTAPI DEPLOYMENT SETUP

Checking working directory...
✓ Working directory confirmed.
  C:\Users\EUGENE\Desktop\RISK MODELLING ANALYTICS\CUSTOMER ANALYTICS\PayStone _Explainable AI Bank Customer Churn Prediction

✓ Model directory created/verified:
  C:\Users\EUGENE\Desktop\RISK MODELLING ANALYTICS\CUSTOMER ANALYTICS\PayStone _Explainable AI Bank Customer Churn Prediction\paystone-churn-api\model

Checking saved CatBoost model...
✓ CatBoost model found.
  C:\Users\EUGENE\Desktop\RISK MODELLING ANALYTICS\CUSTOMER ANALYTICS\PayStone _Explainable AI Bank Customer Churn Prediction\CatBoost_churn_model.pkl

Copying CatBoost model...
✓ CatBoost model already exists in the deployment folder.

Testing copied CatBoost model...
✓ CatBoost model loaded successfully.
  Model type: <class 'catboost.core.CatBoostClassifier'>

Creating deployment files...
✓ Already exists: app.py
✓ Already exists: requirements.txt
✓ Already exists: Dockerfile
✓ Already exists: .gitignore


PAYSTONE FASTAPI 

# Automatic CatBoost Feature Extraction & Deployment Schema

In [4]:
# =============================================================================
# PAYSTONE CUSTOMER CHURN PREDICTION
# AUTOMATIC CATBOOST FEATURE EXTRACTION
# AND FASTAPI DEPLOYMENT SCHEMA CREATION
# =============================================================================
#
# PROJECT:
# -------
# PayStone - Explainable AI Bank Customer Churn Prediction
#
# PURPOSE:
# --------
# This pipeline prepares the trained CatBoost model for deployment as a
# FastAPI prediction service.
#
# The trained machine learning model already exists as:
#
#     CatBoost_churn_model.pkl
#
# and has already been loaded into the notebook as:
#
#     tuned_catboost
#
# This script DOES NOT retrain the model.
#
# Instead, it:
#
# 1. Locates the FastAPI deployment directory.
# 2. Confirms that the trained CatBoost model is loaded.
# 3. Extracts the exact feature names used during model training.
# 4. Validates the feature names.
# 5. Checks for duplicate or invalid feature names.
# 6. Counts the number of model input variables.
# 7. Creates a deployment feature schema.
# 8. Saves the schema as feature_columns.json.
# 9. Reloads the JSON file to verify that it was saved correctly.
# 10. Displays the final FastAPI deployment structure.
#
#
# WHY THIS IS IMPORTANT:
# ----------------------
# A deployed machine learning API must receive the same variables that were
# used when the model was trained.
#
# For example, if the CatBoost model was trained using:
#
#     feature_A
#     feature_B
#     feature_C
#
# the API should not accidentally receive:
#
#     feature_A
#     feature_C
#     feature_B
#
# or omit feature_B.
#
# The feature_columns.json file provides a machine-readable record of the
# model's expected input variables.
#
#
# FINAL DEPLOYMENT STRUCTURE:
# ---------------------------
#
# paystone-churn-api/
# │
# ├── app.py
# ├── requirements.txt
# ├── Dockerfile
# ├── .gitignore
# │
# └── model/
#     ├── CatBoost_churn_model.pkl
#     └── feature_columns.json
#
#
# DEPLOYMENT FLOW:
# ----------------
#
# Trained CatBoost Model
#          │
#          ▼
# CatBoost_churn_model.pkl
#          │
#          ▼
# Extract Feature Names
#          │
#          ▼
# feature_columns.json
#          │
#          ▼
# FastAPI app.py
#          │
#          ▼
# Validate incoming customer data
#          │
#          ▼
# CatBoost prediction
#          │
#          ▼
# Churn probability
#          │
#          ▼
# Render Cloud Deployment
#
# =============================================================================


# =============================================================================
# STEP 1: IMPORT REQUIRED LIBRARIES
# =============================================================================
#
# pathlib
# -------
# Used to create and manage Windows file paths.
# It is safer and cleaner than manually joining strings with "\\".
#
# json
# ----
# Used to save the model's feature information in a JSON file.
#
# =============================================================================
# =============================================================================
# STEP 2: DEFINE THE PAYSTONE WORKING DIRECTORY
# =============================================================================
#
# This is the existing folder containing your PayStone machine learning
# project.
#
# The 'r' before the string means "raw string".
#
# This is important on Windows because file paths contain backslashes.
#
# Example:
#
# C:\Users\EUGENE\Desktop\...
#
# Without a raw string, Python can interpret certain backslash combinations
# as escape characters.
#
# =============================================================================

working_dir = Path(
    r"C:\Users\EUGENE\Desktop\RISK MODELLING ANALYTICS\CUSTOMER ANALYTICS\PayStone _Explainable AI Bank Customer Churn Prediction"
)


# =============================================================================
# STEP 3: DEFINE THE FASTAPI DEPLOYMENT DIRECTORY
# =============================================================================
#
# We create a separate folder called:
#
#     paystone-churn-api
#
# inside the existing PayStone project.
#
# This keeps the deployment files separate from the original modelling
# notebook and modelling outputs.
#
# =============================================================================

project_dir = (
    working_dir / "paystone-churn-api"
)


# =============================================================================
# STEP 4: DEFINE THE MODEL DIRECTORY
# =============================================================================
#
# The trained CatBoost model and feature schema will be stored in:
#
#     paystone-churn-api/model/
#
# This provides a clean separation between:
#
#     API code
#     model files
#     deployment configuration
#
# =============================================================================

model_dir = (
    project_dir / "model"
)


# =============================================================================
# STEP 5: DEFINE THE FEATURE SCHEMA FILE
# =============================================================================
#
# The feature schema will be saved as:
#
#     feature_columns.json
#
# This JSON file will contain:
#
#     - Model type
#     - Model filename
#     - Number of expected features
#     - Exact feature names
#
# Example:
#
# {
#     "model": "CatBoost",
#     "model_file": "CatBoost_churn_model.pkl",
#     "number_of_features": 10,
#     "feature_columns": [
#         "feature_1",
#         "feature_2",
#         "feature_3"
#     ]
# }
#
# =============================================================================

feature_file = (
    model_dir / "feature_columns.json"
)


# =============================================================================
# STEP 6: CHECK THAT THE FASTAPI PROJECT DIRECTORY EXISTS
# =============================================================================
#
# The deployment directory should have been created by the previous setup
# script.
#
# If it does not exist, the script stops and tells us to run the folder
# creation pipeline first.
#
# This prevents the script from accidentally creating files in an incorrect
# location.
#
# =============================================================================

if not project_dir.exists():

    raise FileNotFoundError(

        "\nThe FastAPI deployment directory does not exist.\n"

        f"Expected location:\n{project_dir}\n\n"

        "Please run the PayStone FastAPI deployment "
        "folder setup code first."
    )


# =============================================================================
# STEP 7: CREATE / VERIFY THE MODEL DIRECTORY
# =============================================================================
#
# Even though the model directory should already exist, we use:
#
#     mkdir(..., exist_ok=True)
#
# to ensure that it exists before attempting to save the JSON file.
#
# =============================================================================

model_dir.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# STEP 8: DISPLAY PIPELINE HEADER
# =============================================================================

print("=" * 80)

print(
    "PAYSTONE CATBOOST FEATURE EXTRACTION"
)

print("=" * 80)


print(
    "\nFastAPI deployment directory:"
)

print(
    project_dir
)


# =============================================================================
# STEP 9: VERIFY THAT THE TRAINED CATBOOST MODEL IS LOADED
# =============================================================================
#
# The notebook should already contain:
#
#     tuned_catboost = joblib.load(
#         "CatBoost_churn_model.pkl"
#     )
#
# This script does not reload or retrain the model.
#
# Instead, it checks whether the variable:
#
#     tuned_catboost
#
# exists in the current notebook session.
#
# =============================================================================

if "tuned_catboost" not in globals():

    raise NameError(

        "\nThe variable 'tuned_catboost' "
        "is not available in the notebook.\n\n"

        "Please load the trained model first using:\n\n"

        "tuned_catboost = joblib.load("
        "'CatBoost_churn_model.pkl'"
        ")"
    )


# =============================================================================
# STEP 10: CONFIRM MODEL INFORMATION
# =============================================================================
#
# This provides an audit-style confirmation that the correct model object
# has been loaded.
#
# The output will normally look similar to:
#
#     Model type:
#     <class 'catboost.core.CatBoostClassifier'>
#
# =============================================================================

print(
    "\n✓ Trained CatBoost model found in notebook."
)

print(
    f"  Model type: {type(tuned_catboost)}"
)


# =============================================================================
# STEP 11: INITIALISE FEATURE NAME VARIABLE
# =============================================================================
#
# We initially set feature_names to None.
#
# The script will subsequently try to populate this variable using the
# feature information stored in the trained model.
#
# =============================================================================

feature_names = None


# =============================================================================
# STEP 12: EXTRACT CATBOOST FEATURE NAMES
# =============================================================================
#
# CatBoost models normally expose their feature names through:
#
#     feature_names_
#
# These names represent the variables associated with the trained model.
#
# This is our preferred source because it comes directly from the trained
# CatBoost model.
#
# =============================================================================

if hasattr(
    tuned_catboost,
    "feature_names_"
):

    feature_names = list(
        tuned_catboost.feature_names_
    )


    print(
        "\n✓ Feature names extracted successfully."
    )

    print(
        "  Source: tuned_catboost.feature_names_"
    )


# =============================================================================
# STEP 13: FALLBACK TO SCIKIT-LEARN FEATURE NAMES
# =============================================================================
#
# Some models or pipelines may expose feature names through:
#
#     feature_names_in_
#
# Therefore, if CatBoost's feature_names_ is not available, we check this
# alternative attribute.
#
# This makes the feature extraction process more robust.
#
# =============================================================================

elif hasattr(
    tuned_catboost,
    "feature_names_in_"
):

    feature_names = list(
        tuned_catboost.feature_names_in_
    )


    print(
        "\n✓ Feature names extracted successfully."
    )

    print(
        "  Source: tuned_catboost.feature_names_in_"
    )


# =============================================================================
# STEP 14: STOP IF FEATURE NAMES CANNOT BE FOUND
# =============================================================================
#
# If neither feature_names_ nor feature_names_in_ exists, we cannot safely
# determine the API input schema from the saved model alone.
#
# In that situation, the correct alternative is to use the original training
# dataset columns, for example:
#
#     X_train.columns
#
# We deliberately stop rather than guessing the feature names.
#
# =============================================================================

else:

    raise AttributeError(

        "\nUnable to automatically extract feature names "
        "from the trained model.\n\n"

        "Please provide the original training feature columns, "
        "for example:\n\n"

        "X_train.columns"
    )


# =============================================================================
# STEP 15: VALIDATE THAT FEATURES WERE EXTRACTED
# =============================================================================
#
# An empty feature list would indicate that something went wrong during
# model extraction.
#
# A machine learning model cannot be safely exposed through an API without
# knowing its expected input variables.
#
# =============================================================================

if len(feature_names) == 0:

    raise ValueError(

        "The trained model returned zero feature names."
    )


# =============================================================================
# STEP 16: CHECK FOR EMPTY OR INVALID FEATURE NAMES
# =============================================================================
#
# This checks whether any feature is:
#
#     None
#     ""
#     "   "
#
# Invalid feature names could cause problems when the API receives customer
# prediction data.
#
# =============================================================================

invalid_features = [

    feature

    for feature in feature_names

    if feature is None

    or str(feature).strip() == ""

]


if invalid_features:

    raise ValueError(

        "Invalid or empty feature names detected:\n"

        f"{invalid_features}"
    )


# =============================================================================
# STEP 17: CONVERT FEATURE NAMES TO STRINGS
# =============================================================================
#
# JSON works most reliably with strings.
#
# Therefore, we convert every feature name into a string before saving the
# deployment schema.
#
# =============================================================================

feature_names = [

    str(feature)

    for feature in feature_names

]


# =============================================================================
# STEP 18: CHECK FOR DUPLICATE FEATURE NAMES
# =============================================================================
#
# Every model input variable should have a unique name.
#
# For example:
#
#     customer_age
#     customer_age
#
# would be invalid for our deployment schema.
#
# Duplicate names could create ambiguity when the API receives JSON input.
#
# =============================================================================

duplicate_features = [

    feature

    for feature in set(feature_names)

    if feature_names.count(feature) > 1

]


if duplicate_features:

    raise ValueError(

        "\nDuplicate feature names detected:\n"

        f"{duplicate_features}"
    )


# =============================================================================
# STEP 19: COUNT THE MODEL FEATURES
# =============================================================================
#
# The number of features tells the FastAPI application how many input
# variables the model expects.
#
# Example:
#
#     20 features
#
# means that a valid prediction request should contain the required 20
# variables.
#
# =============================================================================

number_of_features = len(
    feature_names
)


# =============================================================================
# STEP 20: DISPLAY MODEL FEATURE INFORMATION
# =============================================================================

print("\n")

print("=" * 80)

print(
    "MODEL FEATURE INFORMATION"
)

print("=" * 80)


print(
    f"\nNumber of model features: "
    f"{number_of_features}"
)


# =============================================================================
# STEP 21: DISPLAY EACH FEATURE
# =============================================================================
#
# This creates an easy-to-read audit list of the variables used by the model.
#
# The numbering also makes it easier to compare these features against:
#
#     X_train.columns
#     X_test.columns
#     feature importance output
#     SHAP analysis
#     FastAPI input schema
#
# =============================================================================

print(
    "\nFeature names:"
)


for index, feature in enumerate(

    feature_names,

    start=1

):

    print(
        f"{index:>4}. {feature}"
    )


# =============================================================================
# STEP 22: CREATE THE DEPLOYMENT FEATURE SCHEMA
# =============================================================================
#
# We now create a Python dictionary containing the information required by
# the FastAPI application.
#
# This creates a single source of truth for the model's expected inputs.
#
# =============================================================================

feature_schema = {

    # Machine learning algorithm
    "model":
        "CatBoost",


    # Name of the saved model file
    "model_file":
        "CatBoost_churn_model.pkl",


    # Number of expected input variables
    "number_of_features":
        number_of_features,


    # Exact model feature names
    "feature_columns":
        feature_names

}


# =============================================================================
# STEP 23: SAVE THE FEATURE SCHEMA AS JSON
# =============================================================================
#
# json.dump() writes the Python dictionary to:
#
#     model/feature_columns.json
#
# indent=4 makes the file easy for humans to read and inspect.
#
# encoding="utf-8" ensures that feature names are saved correctly.
#
# =============================================================================

with open(

    feature_file,

    "w",

    encoding="utf-8"

) as file:

    json.dump(

        feature_schema,

        file,

        indent=4
    )


# =============================================================================
# STEP 24: CONFIRM JSON FILE CREATION
# =============================================================================

print("\n")

print("=" * 80)

print(
    "FEATURE SCHEMA SAVED SUCCESSFULLY"
)

print("=" * 80)


print(
    f"\nSaved file:"
)

print(
    feature_file
)


# =============================================================================
# STEP 25: RELOAD THE JSON FILE
# =============================================================================
#
# We now read the file back from disk.
#
# This is a deployment-quality validation step.
#
# It confirms that:
#
#     1. The file exists.
#     2. The JSON is valid.
#     3. The saved feature information can be accessed.
#
# =============================================================================

with open(

    feature_file,

    "r",

    encoding="utf-8"

) as file:

    saved_schema = json.load(
        file
    )


# =============================================================================
# STEP 26: EXTRACT SAVED FEATURE INFORMATION
# =============================================================================
#
# Retrieve the feature list and feature count from the saved JSON file.
#
# =============================================================================

saved_features = (
    saved_schema["feature_columns"]
)


saved_feature_count = (
    saved_schema["number_of_features"]
)


# =============================================================================
# STEP 27: VALIDATE FEATURE COUNT
# =============================================================================
#
# The number stored in JSON must equal the actual number of feature names.
#
# Example:
#
#     number_of_features = 20
#
# must correspond to:
#
#     len(feature_columns) = 20
#
# =============================================================================

if (

    saved_feature_count

    !=

    len(saved_features)

):

    raise ValueError(

        "Feature count validation failed.\n"

        f"Stored count: {saved_feature_count}\n"

        f"Actual count: {len(saved_features)}"
    )


# =============================================================================
# STEP 28: VALIDATE FEATURE NAMES
# =============================================================================
#
# The feature list read from the JSON file must exactly match the feature
# list extracted from the trained model.
#
# This prevents accidental modification of the deployment schema.
#
# =============================================================================

if saved_features != feature_names:

    raise ValueError(

        "Saved feature names do not match "
        "the extracted model features."
    )


# =============================================================================
# STEP 29: CONFIRM SCHEMA VALIDATION
# =============================================================================

print(
    "\n✓ Saved feature schema verified successfully."
)


# =============================================================================
# STEP 30: DISPLAY UPDATED DEPLOYMENT STRUCTURE
# =============================================================================
#
# The following section recursively searches the FastAPI deployment folder
# and displays all files.
#
# This allows us to confirm that:
#
#     app.py
#     requirements.txt
#     Dockerfile
#     .gitignore
#     CatBoost_churn_model.pkl
#     feature_columns.json
#
# are available.
#
# =============================================================================

print("\n")

print("=" * 80)

print(
    "UPDATED PAYSTONE FASTAPI DEPLOYMENT STRUCTURE"
)

print("=" * 80)


for path in sorted(

    project_dir.rglob("*")

):

    if path.is_file():

        print(

            "   ✓",

            path.relative_to(
                project_dir
            )

        )


# =============================================================================
# STEP 31: FINAL DEPLOYMENT STATUS
# =============================================================================

print("\n")

print("=" * 80)

print(
    "FEATURE EXTRACTION COMPLETED SUCCESSFULLY"
)

print("=" * 80)


print(
    "\nFeature schema available at:"
)


print(
    feature_file
)


print(
    "\nThe FastAPI application can now use this "
    "schema to validate incoming customer data "
    "against the exact features used by the "
    "trained CatBoost model."
)


# =============================================================================
# END OF FEATURE EXTRACTION PIPELINE
# =============================================================================
#
# NEXT DEPLOYMENT STAGE:
#
#     feature_columns.json
#             │
#             ▼
#         app.py
#             │
#             ▼
#      FastAPI /predict
#             │
#             ▼
#      Input validation
#             │
#             ▼
#     CatBoost prediction
#             │
#             ▼
#      Churn probability
#             │
#             ▼
#        Dockerfile
#             │
#             ▼
#          GitHub
#             │
#             ▼
#          Render
#
# =============================================================================

PAYSTONE CATBOOST FEATURE EXTRACTION

FastAPI deployment directory:
C:\Users\EUGENE\Desktop\RISK MODELLING ANALYTICS\CUSTOMER ANALYTICS\PayStone _Explainable AI Bank Customer Churn Prediction\paystone-churn-api

✓ Trained CatBoost model found in notebook.
  Model type: <class 'catboost.core.CatBoostClassifier'>

✓ Feature names extracted successfully.
  Source: tuned_catboost.feature_names_


MODEL FEATURE INFORMATION

Number of model features: 25

Feature names:
   1. Income_Category
   2. Card_Category
   3. Education_Level_Doctorate
   4. Education_Level_Graduate
   5. Education_Level_High School
   6. Education_Level_Post-Graduate
   7. Education_Level_Uneducated
   8. Education_Level_Unknown
   9. Marital_Status_Married
  10. Marital_Status_Single
  11. Marital_Status_Unknown
  12. Customer_Age
  13. Gender
  14. Dependent_count
  15. Months_on_book
  16. Total_Relationship_Count
  17. Transaction_Amount_Change_Q4_Q1
  18. Transaction_Count_Change_Q4_Q1
  19. Avg_Utilization_Ratio

# FastAPI Prediction Application

In [5]:
# =============================================================================
# PAYSTONE CUSTOMER CHURN PREDICTION
# FASTAPI APPLICATION BUILDING PIPELINE
# =============================================================================
#
# PURPOSE
# -------
# This pipeline creates the FastAPI application for the PayStone CatBoost
# customer churn prediction model.
#
# IMPORTANT PATH HANDLING
# -----------------------
# This notebook is being executed in Jupyter.
#
# Jupyter notebooks do not normally define the special Python variable:
#
#     __file__
#
# Therefore, we MUST NOT use:
#
#     Path(__file__).resolve().parent
#
# directly inside the notebook.
#
# Instead:
#
#     JUPYTER NOTEBOOK
#     ----------------
#     Use the known Windows deployment directory.
#
#     FASTAPI app.py
#     -------------
#     Use Path(__file__).resolve().parent.
#
# This allows the same app.py to work locally and on Render.
#
# =============================================================================


# =============================================================================
# STEP 1: IMPORT REQUIRED LIBRARIES
# =============================================================================


# =============================================================================
# STEP 2: DEFINE PAYSTONE DEPLOYMENT DIRECTORY
# =============================================================================
#
# IMPORTANT:
#
# This code is running inside Jupyter Notebook.
#
# Therefore, __file__ is NOT available.
#
# We use the actual Windows deployment directory instead.
#
# =============================================================================

BASE_DIR = Path(
    r"C:\Users\EUGENE\Desktop\RISK MODELLING ANALYTICS\CUSTOMER ANALYTICS\PayStone _Explainable AI Bank Customer Churn Prediction\paystone-churn-api"
)


# =============================================================================
# STEP 3: DEFINE MODEL DIRECTORY
# =============================================================================
#
# The trained CatBoost model and feature schema are stored inside:
#
#     paystone-churn-api/
#     └── model/
#
# =============================================================================

MODEL_DIR = BASE_DIR / "model"


# =============================================================================
# STEP 4: DEFINE CATBOOST MODEL PATH
# =============================================================================
#
# This points to the saved trained CatBoost model.
#
# =============================================================================

MODEL_PATH = MODEL_DIR / "CatBoost_churn_model.pkl"


# =============================================================================
# STEP 5: DEFINE FEATURE SCHEMA PATH
# =============================================================================
#
# feature_columns.json contains the exact feature names used by the trained
# CatBoost model.
#
# =============================================================================

FEATURE_SCHEMA_PATH = MODEL_DIR / "feature_columns.json"


# =============================================================================
# STEP 6: DISPLAY PATH INFORMATION
# =============================================================================

print("=" * 80)
print("PAYSTONE FASTAPI APPLICATION PATH CONFIGURATION")
print("=" * 80)

print("\nFastAPI deployment directory:")
print(BASE_DIR)

print("\nModel directory:")
print(MODEL_DIR)

print("\nCatBoost model:")
print(MODEL_PATH)

print("\nFeature schema:")
print(FEATURE_SCHEMA_PATH)


# =============================================================================
# STEP 7: VERIFY DEPLOYMENT DIRECTORY
# =============================================================================
#
# Stop the pipeline if the deployment folder does not exist.
#
# This prevents the application from being created in the wrong location.
#
# =============================================================================

if not BASE_DIR.exists():

    raise FileNotFoundError(

        "\nPayStone FastAPI deployment directory was not found.\n\n"

        f"Expected location:\n{BASE_DIR}\n\n"

        "Please run the FastAPI deployment folder setup "
        "pipeline first."
    )


print("\n✓ FastAPI deployment directory confirmed.")


# =============================================================================
# STEP 8: VERIFY MODEL DIRECTORY
# =============================================================================

if not MODEL_DIR.exists():

    raise FileNotFoundError(

        "\nModel directory was not found.\n\n"

        f"Expected location:\n{MODEL_DIR}"
    )


print("✓ Model directory confirmed.")


# =============================================================================
# STEP 9: VERIFY CATBOOST MODEL
# =============================================================================

if not MODEL_PATH.exists():

    raise FileNotFoundError(

        "\nCatBoost model was not found.\n\n"

        f"Expected location:\n{MODEL_PATH}"
    )


print("✓ CatBoost model found.")


# =============================================================================
# STEP 10: VERIFY FEATURE SCHEMA
# =============================================================================

if not FEATURE_SCHEMA_PATH.exists():

    raise FileNotFoundError(

        "\nFeature schema was not found.\n\n"

        f"Expected location:\n{FEATURE_SCHEMA_PATH}\n\n"

        "Run the automatic CatBoost feature extraction pipeline first."
    )


print("✓ Feature schema found.")


# =============================================================================
# STEP 11: LOAD FEATURE SCHEMA
# =============================================================================
#
# Read feature_columns.json.
#
# This file was automatically created from the trained CatBoost model.
#
# =============================================================================

with open(

    FEATURE_SCHEMA_PATH,

    "r",

    encoding="utf-8"

) as file:

    feature_schema = json.load(file)


# Extract exact feature names
FEATURE_COLUMNS = feature_schema["feature_columns"]


# Extract expected number of features
NUMBER_OF_FEATURES = feature_schema["number_of_features"]


# =============================================================================
# STEP 12: VALIDATE FEATURE SCHEMA
# =============================================================================
#
# Confirm that the recorded feature count agrees with the actual feature list.
#
# =============================================================================

if NUMBER_OF_FEATURES != len(FEATURE_COLUMNS):

    raise ValueError(

        "Feature schema validation failed.\n"

        f"Expected number: {NUMBER_OF_FEATURES}\n"

        f"Actual number: {len(FEATURE_COLUMNS)}"
    )


print(
    f"\n✓ Feature schema validated."
)

print(
    f"✓ Number of expected features: {NUMBER_OF_FEATURES}"
)


# =============================================================================
# STEP 13: LOAD CATBOOST MODEL
# =============================================================================
#
# The trained model is loaded once.
#
# We do not retrain the model during deployment.
#
# =============================================================================

tuned_catboost = joblib.load(
    MODEL_PATH
)


print(
    "\n✓ CatBoost model loaded successfully."
)

print(
    f"  Model type: {type(tuned_catboost)}"
)


# =============================================================================
# STEP 14: CREATE FASTAPI APPLICATION CODE
# =============================================================================
#
# IMPORTANT:
#
# The following code will be written into:
#
#     paystone-churn-api/app.py
#
# The app.py file will use:
#
#     Path(__file__).resolve().parent
#
# rather than the Windows path above.
#
# This is what makes the final application portable to Render.
#
# =============================================================================

app_code = r'''
# =============================================================================
# PAYSTONE CUSTOMER CHURN PREDICTION
# FASTAPI PREDICTION APPLICATION
# =============================================================================
#
# This application exposes the trained PayStone CatBoost churn model through
# a RESTful API.
#
# ENDPOINTS
# ---------
#
# GET  /health
# GET  /model-info
# POST /predict
# GET  /docs
# GET  /redoc
#
# =============================================================================


# =============================================================================
# STEP 1: IMPORT REQUIRED LIBRARIES
# =============================================================================



# =============================================================================
# STEP 2: DEFINE APPLICATION PATH
# =============================================================================
#
# IMPORTANT:
#
# Unlike the Jupyter Notebook, app.py DOES have access to __file__.
#
# Therefore, we use __file__ here.
#
# This makes the application portable across:
#
#     Windows
#     GitHub
#     Render
#
# The application automatically finds the model folder relative to app.py.
#
# =============================================================================

BASE_DIR = Path(__file__).resolve().parent


# =============================================================================
# STEP 3: DEFINE MODEL DIRECTORY
# =============================================================================

MODEL_DIR = BASE_DIR / "model"


# =============================================================================
# STEP 4: DEFINE CATBOOST MODEL PATH
# =============================================================================

MODEL_PATH = MODEL_DIR / "CatBoost_churn_model.pkl"


# =============================================================================
# STEP 5: DEFINE FEATURE SCHEMA PATH
# =============================================================================

FEATURE_SCHEMA_PATH = MODEL_DIR / "feature_columns.json"


# =============================================================================
# STEP 6: VERIFY FEATURE SCHEMA
# =============================================================================

if not FEATURE_SCHEMA_PATH.exists():

    raise FileNotFoundError(
        f"Feature schema not found: {FEATURE_SCHEMA_PATH}"
    )


# =============================================================================
# STEP 7: LOAD FEATURE SCHEMA
# =============================================================================

with open(
    FEATURE_SCHEMA_PATH,
    "r",
    encoding="utf-8"
) as file:

    feature_schema = json.load(file)


FEATURE_COLUMNS = feature_schema["feature_columns"]

NUMBER_OF_FEATURES = feature_schema["number_of_features"]


# =============================================================================
# STEP 8: VALIDATE FEATURE SCHEMA
# =============================================================================

if NUMBER_OF_FEATURES != len(FEATURE_COLUMNS):

    raise ValueError(
        "Feature schema validation failed."
    )


# =============================================================================
# STEP 9: LOAD CATBOOST MODEL
# =============================================================================

if not MODEL_PATH.exists():

    raise FileNotFoundError(
        f"CatBoost model not found: {MODEL_PATH}"
    )


tuned_catboost = joblib.load(
    MODEL_PATH
)


# =============================================================================
# STEP 10: CREATE FASTAPI APPLICATION
# =============================================================================

app = FastAPI(

    title="PayStone Customer Churn Prediction API",

    description=(
        "REST API for predicting bank customer churn "
        "using a trained CatBoost model."
    ),

    version="1.0.0"
)


# =============================================================================
# STEP 11: DEFINE CUSTOMER REQUEST FORMAT
# =============================================================================

class CustomerData(BaseModel):

    data: Dict[str, Any]


# =============================================================================
# STEP 12: HEALTH CHECK ENDPOINT
# =============================================================================
#
# GET /health
#
# Used to confirm that the API and model are operational.
#
# =============================================================================

@app.get("/health")
def health_check():

    return {

        "status": "healthy",

        "model_loaded": tuned_catboost is not None,

        "model": "CatBoost",

        "number_of_features": NUMBER_OF_FEATURES

    }


# =============================================================================
# STEP 13: MODEL INFORMATION ENDPOINT
# =============================================================================
#
# GET /model-info
#
# Returns information about the deployed model.
#
# =============================================================================

@app.get("/model-info")
def model_info():

    return {

        "model_type": "CatBoost",

        "model_file": MODEL_PATH.name,

        "number_of_features": NUMBER_OF_FEATURES,

        "feature_columns": FEATURE_COLUMNS,

        "api_version": "1.0.0"

    }


# =============================================================================
# STEP 14: CUSTOMER CHURN PREDICTION ENDPOINT
# =============================================================================
#
# POST /predict
#
# Receives customer information and returns:
#
#     - prediction
#     - churn prediction
#     - churn probability
#     - no-churn probability
#
# =============================================================================

@app.post("/predict")
def predict_churn(customer: CustomerData):


    # =========================================================================
    # STEP 14.1: EXTRACT CUSTOMER DATA
    # =========================================================================

    customer_data = customer.data


    # =========================================================================
    # STEP 14.2: CHECK FOR MISSING FEATURES
    # =========================================================================

    missing_features = [

        feature

        for feature in FEATURE_COLUMNS

        if feature not in customer_data

    ]


    if missing_features:

        raise HTTPException(

            status_code=400,

            detail={

                "error": "Missing required features",

                "missing_features": missing_features

            }

        )


    # =========================================================================
    # STEP 14.3: CHECK FOR UNEXPECTED FEATURES
    # =========================================================================

    unexpected_features = [

        feature

        for feature in customer_data

        if feature not in FEATURE_COLUMNS

    ]


    if unexpected_features:

        raise HTTPException(

            status_code=400,

            detail={

                "error": "Unexpected features supplied",

                "unexpected_features": unexpected_features

            }

        )


    # =========================================================================
    # STEP 14.4: ARRANGE FEATURES IN MODEL ORDER
    # =========================================================================

    ordered_data = {

        feature: customer_data[feature]

        for feature in FEATURE_COLUMNS

    }


    # =========================================================================
    # STEP 14.5: CREATE PREDICTION DATAFRAME
    # =========================================================================

    input_df = pd.DataFrame(

        [ordered_data],

        columns=FEATURE_COLUMNS

    )


    # =========================================================================
    # STEP 14.6: RUN CATBOOST PREDICTION
    # =========================================================================

    try:

        prediction = tuned_catboost.predict(
            input_df
        )

        probabilities = tuned_catboost.predict_proba(
            input_df
        )

    except Exception as error:

        raise HTTPException(

            status_code=500,

            detail={

                "error": "Model prediction failed",

                "message": str(error)

            }

        )


    # =========================================================================
    # STEP 14.7: EXTRACT PREDICTION RESULTS
    # =========================================================================

    predicted_class = int(
        prediction[0]
    )


    no_churn_probability = float(
        probabilities[0][0]
    )


    churn_probability = float(
        probabilities[0][1]
    )


    # =========================================================================
    # STEP 14.8: CREATE BUSINESS-FRIENDLY LABEL
    # =========================================================================

    if predicted_class == 1:

        churn_prediction = "Churn"

    else:

        churn_prediction = "No Churn"


    # =========================================================================
    # STEP 14.9: RETURN PREDICTION
    # =========================================================================

    return {

        "prediction": predicted_class,

        "churn_prediction": churn_prediction,

        "churn_probability": round(
            churn_probability,
            4
        ),

        "no_churn_probability": round(
            no_churn_probability,
            4
        )

    }


# =============================================================================
# END OF FASTAPI APPLICATION
# =============================================================================
'''


# =============================================================================
# STEP 15: SAVE app.py
# =============================================================================
#
# The generated application is saved directly inside the FastAPI deployment
# directory.
#
# =============================================================================

app_file = BASE_DIR / "app.py"


with open(

    app_file,

    "w",

    encoding="utf-8"

) as file:

    file.write(app_code)


# =============================================================================
# STEP 16: VERIFY app.py
# =============================================================================

if not app_file.exists():

    raise FileNotFoundError(
        f"app.py was not created: {app_file}"
    )


print("\n")

print("=" * 80)

print(
    "FASTAPI APPLICATION CREATED SUCCESSFULLY"
)

print("=" * 80)

print(
    f"\napp.py location:\n{app_file}"
)


# =============================================================================
# STEP 17: DISPLAY FINAL DEPLOYMENT STRUCTURE
# =============================================================================

print("\n")

print("=" * 80)

print(
    "PAYSTONE FASTAPI DEPLOYMENT STRUCTURE"
)

print("=" * 80)


for path in sorted(BASE_DIR.rglob("*")):

    if path.is_file():

        print(
            "   ✓",
            path.relative_to(BASE_DIR)
        )


# =============================================================================
# STEP 18: FINAL STATUS
# =============================================================================

print("\n")

print("=" * 80)

print(
    "STEP 4 - FASTAPI APPLICATION BUILD COMPLETED"
)

print("=" * 80)

print(
    "\nThe application is now ready for local testing."
)

print(
    "\nNext steps:"
)

print(
    "1. Create requirements.txt"
)

print(
    "2. Create Dockerfile"
)

print(
    "3. Start FastAPI with Uvicorn"
)

print(
    "4. Test /health"
)

print(
    "5. Test /model-info"
)

print(
    "6. Test /predict"
)

print(
    "7. Test Swagger at /docs"
)

print(
    "8. Push the project to GitHub"
)

print(
    "9. Deploy to Render"
)

PAYSTONE FASTAPI APPLICATION PATH CONFIGURATION

FastAPI deployment directory:
C:\Users\EUGENE\Desktop\RISK MODELLING ANALYTICS\CUSTOMER ANALYTICS\PayStone _Explainable AI Bank Customer Churn Prediction\paystone-churn-api

Model directory:
C:\Users\EUGENE\Desktop\RISK MODELLING ANALYTICS\CUSTOMER ANALYTICS\PayStone _Explainable AI Bank Customer Churn Prediction\paystone-churn-api\model

CatBoost model:
C:\Users\EUGENE\Desktop\RISK MODELLING ANALYTICS\CUSTOMER ANALYTICS\PayStone _Explainable AI Bank Customer Churn Prediction\paystone-churn-api\model\CatBoost_churn_model.pkl

Feature schema:
C:\Users\EUGENE\Desktop\RISK MODELLING ANALYTICS\CUSTOMER ANALYTICS\PayStone _Explainable AI Bank Customer Churn Prediction\paystone-churn-api\model\feature_columns.json

✓ FastAPI deployment directory confirmed.
✓ Model directory confirmed.
✓ CatBoost model found.
✓ Feature schema found.

✓ Feature schema validated.
✓ Number of expected features: 25

✓ CatBoost model loaded successfully.
  Model typ

# Creates Requirements Text and Dockerfile

In [6]:
# =============================================================================
# PAYSTONE CUSTOMER CHURN PREDICTION
# STEP 5: CREATE DEPLOYMENT REQUIREMENTS AND DOCKERFILE
# =============================================================================
#
# PURPOSE
# -------
# This pipeline automatically creates:
#
#     1. requirements.txt
#     2. Dockerfile
#
# It then verifies that the required deployment files exist and performs
# basic local validation before the application is pushed to GitHub and
# deployed to Render.
#
#
# DEPLOYMENT FLOW
# ---------------
#
# CatBoost Model
#       ↓
# FastAPI app.py
#       ↓
# requirements.txt
#       ↓
# Dockerfile
#       ↓
# Local API Testing
#       ↓
# GitHub
#       ↓
# Render
#
# =============================================================================


# =============================================================================
# STEP 1: IMPORT REQUIRED LIBRARIES
# =============================================================================

# Path: manages deployment file paths
from pathlib import Path

# JSON: verifies the feature schema
import json


# =============================================================================
# STEP 2: DEFINE PAYSTONE DEPLOYMENT DIRECTORY
# =============================================================================
#
# IMPORTANT:
#
# This code is running inside Jupyter Notebook.
#
# Therefore, we use the actual Windows deployment directory rather than
# __file__.
#
# =============================================================================

BASE_DIR = Path(
    r"C:\Users\EUGENE\Desktop\RISK MODELLING ANALYTICS\CUSTOMER ANALYTICS\PayStone _Explainable AI Bank Customer Churn Prediction\paystone-churn-api"
)


# =============================================================================
# STEP 3: DEFINE DEPLOYMENT FILE PATHS
# =============================================================================

APP_FILE = BASE_DIR / "app.py"

REQUIREMENTS_FILE = BASE_DIR / "requirements.txt"

DOCKERFILE = BASE_DIR / "Dockerfile"

MODEL_DIR = BASE_DIR / "model"

MODEL_FILE = MODEL_DIR / "CatBoost_churn_model.pkl"

FEATURE_SCHEMA_FILE = MODEL_DIR / "feature_columns.json"


# =============================================================================
# STEP 4: DISPLAY DEPLOYMENT LOCATION
# =============================================================================

print("=" * 80)

print(
    "PAYSTONE FASTAPI DEPLOYMENT CONFIGURATION"
)

print("=" * 80)

print(
    "\nDeployment directory:"
)

print(
    BASE_DIR
)


# =============================================================================
# STEP 5: VERIFY DEPLOYMENT DIRECTORY
# =============================================================================

if not BASE_DIR.exists():

    raise FileNotFoundError(

        "\nPayStone FastAPI deployment directory "
        "was not found.\n\n"

        f"Expected location:\n{BASE_DIR}"
    )


print(
    "\n✓ Deployment directory confirmed."
)


# =============================================================================
# STEP 6: VERIFY FASTAPI APPLICATION
# =============================================================================
#
# requirements.txt and Dockerfile are only useful if app.py already exists.
#
# =============================================================================

if not APP_FILE.exists():

    raise FileNotFoundError(

        "\napp.py was not found.\n\n"

        f"Expected location:\n{APP_FILE}\n\n"

        "Run the FastAPI application creation pipeline first."
    )


print(
    "✓ app.py found."
)


# =============================================================================
# STEP 7: VERIFY MODEL FILE
# =============================================================================

if not MODEL_FILE.exists():

    raise FileNotFoundError(

        "\nCatBoost model was not found.\n\n"

        f"Expected location:\n{MODEL_FILE}"
    )


print(
    "✓ CatBoost model found."
)


# =============================================================================
# STEP 8: VERIFY FEATURE SCHEMA
# =============================================================================

if not FEATURE_SCHEMA_FILE.exists():

    raise FileNotFoundError(

        "\nFeature schema was not found.\n\n"

        f"Expected location:\n{FEATURE_SCHEMA_FILE}"
    )


print(
    "✓ Feature schema found."
)


# =============================================================================
# STEP 9: READ FEATURE SCHEMA
# =============================================================================
#
# This provides an additional validation that the deployment schema is
# readable before the application is packaged into Docker.
#
# =============================================================================

with open(

    FEATURE_SCHEMA_FILE,

    "r",

    encoding="utf-8"

) as file:

    feature_schema = json.load(file)


FEATURE_COLUMNS = feature_schema["feature_columns"]

NUMBER_OF_FEATURES = feature_schema["number_of_features"]


# =============================================================================
# STEP 10: VALIDATE FEATURE SCHEMA
# =============================================================================

if NUMBER_OF_FEATURES != len(FEATURE_COLUMNS):

    raise ValueError(

        "Feature schema validation failed.\n"

        f"Number recorded in JSON: {NUMBER_OF_FEATURES}\n"

        f"Number of feature columns: {len(FEATURE_COLUMNS)}"
    )


print(
    f"✓ Feature schema validated: "
    f"{NUMBER_OF_FEATURES} features."
)


# =============================================================================
# STEP 11: CREATE requirements.txt
# =============================================================================
#
# requirements.txt tells the deployment environment which Python packages
# are required to run the FastAPI application.
#
# The main dependencies are:
#
#     fastapi
#     uvicorn
#     catboost
#     joblib
#     pandas
#     pydantic
#
# Uvicorn is the ASGI server used to run FastAPI.
#
# =============================================================================

requirements_content = """\
fastapi
uvicorn[standard]
catboost
joblib
pandas
pydantic
"""


# =============================================================================
# STEP 12: SAVE requirements.txt
# =============================================================================

with open(

    REQUIREMENTS_FILE,

    "w",

    encoding="utf-8"

) as file:

    file.write(
        requirements_content
    )


print(
    "\n✓ requirements.txt created."
)

print(
    f"  Location: {REQUIREMENTS_FILE}"
)


# =============================================================================
# STEP 13: CREATE Dockerfile
# =============================================================================
#
# The Dockerfile defines how Render should build the PayStone API.
#
# PROCESS:
#
#     Python base image
#          ↓
#     Install requirements
#          ↓
#     Copy application
#          ↓
#     Copy model
#          ↓
#     Start Uvicorn
#
# IMPORTANT:
#
# Render will run the application inside a Linux container, so we do not
# reference the Windows C:\ path anywhere in this Dockerfile.
#
# =============================================================================

dockerfile_content = """\
# =============================================================================
# PAYSTONE CUSTOMER CHURN PREDICTION
# FASTAPI DOCKER DEPLOYMENT
# =============================================================================

# Use a lightweight Python image
FROM python:3.11-slim

# Prevent Python from creating .pyc files
ENV PYTHONDONTWRITEBYTECODE=1

# Ensure Python output appears immediately in logs
ENV PYTHONUNBUFFERED=1

# Set the working directory inside the container
WORKDIR /app

# Copy dependency file first
# This allows Docker to cache dependency installation.
COPY requirements.txt .

# Install Python dependencies
RUN pip install --no-cache-dir -r requirements.txt

# Copy the FastAPI application
COPY app.py .

# Copy the trained model and feature schema
COPY model ./model

# Expose the FastAPI port
EXPOSE 8000

# Start the FastAPI application using Uvicorn
#
# 0.0.0.0 allows the application to receive external connections.
#
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
"""


# =============================================================================
# STEP 14: SAVE Dockerfile
# =============================================================================

with open(

    DOCKERFILE,

    "w",

    encoding="utf-8"

) as file:

    file.write(
        dockerfile_content
    )


print(
    "✓ Dockerfile created."
)

print(
    f"  Location: {DOCKERFILE}"
)


# =============================================================================
# STEP 15: VERIFY REQUIREMENTS FILE
# =============================================================================

if not REQUIREMENTS_FILE.exists():

    raise FileNotFoundError(
        "requirements.txt was not created."
    )


requirements_text = REQUIREMENTS_FILE.read_text(
    encoding="utf-8"
)


required_packages = [

    "fastapi",

    "uvicorn",

    "catboost",

    "joblib",

    "pandas",

    "pydantic"

]


missing_packages = [

    package

    for package in required_packages

    if package.lower() not in requirements_text.lower()

]


if missing_packages:

    raise ValueError(

        "The following required packages "
        "are missing from requirements.txt:\n"

        f"{missing_packages}"
    )


print(
    "\n✓ requirements.txt validation completed."
)


# =============================================================================
# STEP 16: VERIFY DOCKERFILE
# =============================================================================

if not DOCKERFILE.exists():

    raise FileNotFoundError(
        "Dockerfile was not created."
    )


docker_text = DOCKERFILE.read_text(
    encoding="utf-8"
)


required_docker_items = [

    "FROM python:3.11-slim",

    "COPY requirements.txt",

    "RUN pip install",

    "COPY app.py",

    "COPY model",

    "uvicorn",

    "0.0.0.0",

    "8000"

]


missing_docker_items = [

    item

    for item in required_docker_items

    if item not in docker_text

]


if missing_docker_items:

    raise ValueError(

        "Dockerfile validation failed.\n"

        f"Missing items: {missing_docker_items}"
    )


print(
    "✓ Dockerfile validation completed."
)


# =============================================================================
# STEP 17: DISPLAY FINAL DEPLOYMENT STRUCTURE
# =============================================================================

print("\n")

print("=" * 80)

print(
    "PAYSTONE FASTAPI DEPLOYMENT STRUCTURE"
)

print("=" * 80)


for path in sorted(

    BASE_DIR.rglob("*")

):

    if path.is_file():

        print(

            "   ✓",

            path.relative_to(BASE_DIR)

        )


# =============================================================================
# STEP 18: FINAL STATUS
# =============================================================================

print("\n")

print("=" * 80)

print(
    "STEP 5 - DEPLOYMENT FILE CREATION COMPLETED"
)

print("=" * 80)


print(
    "\nCreated files:"
)

print(
    "✓ app.py"
)

print(
    "✓ requirements.txt"
)

print(
    "✓ Dockerfile"
)

print(
    "✓ model/CatBoost_churn_model.pkl"
)

print(
    "✓ model/feature_columns.json"
)


print(
    "\nNext stage:"
)

print(
    "STEP 6 - LOCAL FASTAPI TESTING"
)

PAYSTONE FASTAPI DEPLOYMENT CONFIGURATION

Deployment directory:
C:\Users\EUGENE\Desktop\RISK MODELLING ANALYTICS\CUSTOMER ANALYTICS\PayStone _Explainable AI Bank Customer Churn Prediction\paystone-churn-api

✓ Deployment directory confirmed.
✓ app.py found.
✓ CatBoost model found.
✓ Feature schema found.
✓ Feature schema validated: 25 features.

✓ requirements.txt created.
  Location: C:\Users\EUGENE\Desktop\RISK MODELLING ANALYTICS\CUSTOMER ANALYTICS\PayStone _Explainable AI Bank Customer Churn Prediction\paystone-churn-api\requirements.txt
✓ Dockerfile created.
  Location: C:\Users\EUGENE\Desktop\RISK MODELLING ANALYTICS\CUSTOMER ANALYTICS\PayStone _Explainable AI Bank Customer Churn Prediction\paystone-churn-api\Dockerfile

✓ requirements.txt validation completed.
✓ Dockerfile validation completed.


PAYSTONE FASTAPI DEPLOYMENT STRUCTURE
   ✓ .gitignore
   ✓ app.py
   ✓ Dockerfile
   ✓ model\CatBoost_churn_model.pkl
   ✓ model\feature_columns.json
   ✓ requirements.txt


STEP 5 - D